In [1]:
%pip install --upgrade openai-agents

     ---------------------------------------- 0.0/108.8 kB ? eta -:--:--
     ---------------------------------------- 0.0/108.8 kB ? eta -:--:--
     --- ------------------------------------ 10.2/108.8 kB ? eta -:--:--
     --- ------------------------------------ 10.2/108.8 kB ? eta -:--:--
     ---------- -------------------------- 30.7/108.8 kB 187.9 kB/s eta 0:00:01
     -------------------- ---------------- 61.4/108.8 kB 297.7 kB/s eta 0:00:01
     ------------------------ ------------ 71.7/108.8 kB 280.5 kB/s eta 0:00:01
     ------------------------ ------------ 71.7/108.8 kB 280.5 kB/s eta 0:00:01
     ------------------------ ------------ 71.7/108.8 kB 280.5 kB/s eta 0:00:01
     ------------------------ ------------ 71.7/108.8 kB 280.5 kB/s eta 0:00:01
     ------------------------ ------------ 71.7/108.8 kB 280.5 kB/s eta 0:00:01
     ------------------------ ------------ 71.7/108.8 kB 280.5 kB/s eta 0:00:01
     ------------------------ ------------ 71.7/108.8 kB 280.5 kB/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.14.2 requires pydantic~=2.11.9, but you have pydantic 2.13.3 which is incompatible.
crewai 1.14.2 requires python-dotenv~=1.1.1, but you have python-dotenv 1.2.2 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
%pip install -q gradio openai python-dotenv requests httpx pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# Define a helper function to display markdown nicely 
def print_markdown(text):
    display(Markdown(text))

In [10]:
# Import libraries for the notebook client
import os
import requests  # For making HTTP requests
import httpx  # An alternative async-friendly HTTP client (good practice)
import json  # For handling JSON data (manifests, action responses)
from dotenv import load_dotenv
from IPython.display import display, Markdown, Image  # To display results nicely
from openai import OpenAI  # or litellm, groq, etc.

from PIL import Image
import asyncio, pathlib

# Load environment variables (needed for the Gradio apps when they run)
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")



In [11]:
#URL where gradio servers will be running
MCP_BASE = "http://localhost:7860/gradio_api/mcp/sse"


# MCPServerSse: A class that sets up a server using Server-Sent Events (SSE) to serve MCP-compatible tools or agents.
# It allows Communication via SSE, which is a way to push updates from server to client (used in real-time AI applications).
# MCPServerSse allows your tool (e.g., an AI model wrapped with Gradio) to be served as an MCP tool endpoint.
from agents.mcp import MCPServerSse 

#timeout 30 seconds is how long server waits for single tool execution
#client 60 secs timeout isn how long client session kept alive w/o inactivity before its considered expired or disconnected
#if client sends nothin in thos 60 secs, sessio times out

mcp_tool = MCPServerSse(
    {
        "name": "AI_Tutor",
        "url":MCP_BASE,
        "timeout":30,
        "client_session_timeout_seconds":60
    }
)

In [12]:
# httpx for modern async-friendly requests (though requests works fine too)
# This code is designed to fetch and display a schema (manifest) from an MCP server

client = httpx.Client() #Create http client instance

def fetch_schema(server_url):
    """fetches and parses the MCP schema from a server"""
    schema_url = server_url.replace("sse", "schema") #From the mcpbase url replace sse w schema, so that we get schema in return
    print(f"fetching schema from: {schema_url}")

    response = client.get(schema_url, timeout = 10)
    response.raise_for_status() #raise exception if 4xx or 5xx
    schema_data = response.json()
    print("Schema fetched successfully")
    return schema_data

In [14]:
# Fetch the manifest from the AI Tutor server

# Note that the schema is used by AI agents to know how to call the AI Tutor tool programmatically, ensuring the inputs match the required format.

print("--- Fetching AI Tutor Schema ---")
tutor_schema = fetch_schema(MCP_BASE)

if tutor_schema:
    print("\nAI Tutor Schema Contents:")
    # Pretty print the JSON manifest
    print(json.dumps(tutor_schema, indent=2))

print("\n" + "=" * 50 + "\n")  # Separator


--- Fetching AI Tutor Schema ---
fetching schema from: http://localhost:7860/gradio_api/mcp/schema
Schema fetched successfully

AI Tutor Schema Contents:
[
  {
    "name": "explain_concept",
    "description": "Stream an explanation of the question for chosen level (1-5) or default",
    "inputSchema": {
      "type": "object",
      "properties": {
        "question": {
          "type": "string",
          "description": ""
        },
        "level": {
          "type": "number",
          "description": "",
          "default": 3
        }
      }
    },
    "meta": {
      "file_data_present": false,
      "mcp_type": "tool",
      "endpoint_name": "explain_concept"
    }
  },
  {
    "name": "summarize_text",
    "description": "Stream a summary of *text* to roughly *compression_ratio* length *compression_ratio b/w 0.1-0.8*",
    "inputSchema": {
      "type": "object",
      "properties": {
        "text": {
          "type": "string",
          "description": ""
        },
    

# CREATE AI AGENT USING OPENAI SDK WHICH USES MCP TOOLS

In [15]:
# Let's Build the AI agent
from agents import Agent, Runner

agent = Agent(
    name = "Smart Assistant",
    instructions = """
    Context
    -------
    You are an AI assistant with access to an MCP server exposing **four streaming tools**:

    1. **explain_concept**  
    Arguments: { "question": <str>, "level": <int 1‑5> }  
    • Streams an explanation of any concept at the requested depth.

    2. **summarize_text**  
    Arguments: { "text": <str>, "compression_ratio": <float 0.1‑0.8> }  
    • Streams a concise summary ~compression_ratio × original length.

    3. **generate_flashcards**  
    Arguments: { "topic": <str>, "num_cards": <int 1‑20> }  
    • Streams JSON‑lines flashcards: one card per line `{ "q":…, "a":… }`.

    4. **quiz_me**  
    Arguments: { "topic": <str>, "level": <int 1‑5>, "num_questions": <int 1‑15> }  
    • Streams an MC‑question quiz, then an ANSWER KEY section.

    Objective
    ---------
    Help users learn by:
    • Explaining concepts at the depth they request.  
    • Summarising long passages.  
    • Generating flashcards for self‑study.  
    • Quizzing them interactively.

    How to respond
    --------------
    • For each user request, decide which tool (if any) fulfils it best.  
    • Call the tool via MCP by returning *only* the JSON with `"tool"` and `"arguments"` (no extra text).  
    • If a follow‑up conversation is needed (e.g., clarification), ask the user first.  
    • If no tool fits, answer directly in plain language.

    Examples
    --------
    User: “Explain quantum tunnelling like I’m 10.”  
    → Call `explain_concept` with { "question": "quantum tunnelling", "level": 2 }

    User: “Summarise this article to 20 %.” + <article text>  
    → Call `summarize_text` with { "text": "...", "compression_ratio": 0.2 }

    Chat capability
    ---------------
    After each tool call completes (streaming back to the user), remain in the chat loop ready for the next user turn.
    """,
    model = "gpt-4o-mini",
    mcp_servers = [mcp_tool],
)

In [16]:
# This code snippet is implementing a conversational loop with an AI agent that uses MCP tools.

# Opens a connection with an MCP tool. 
# This lets our AI agent interact with an external tool (e.g., image generator, calculator, etc.) over a standard protocol using SSE (Server-Sent Events).
await mcp_tool.connect()  # open SSE channels

result = None
while True:
    user_input = input("User: ")
    if user_input.lower() in {"exit", "quit"}:
        break
        
    # If there was a previous interaction (result is not None), it appends the new user message to the past messages (maintaining conversation context).
    if result is not None:
        new_input = result.to_input_list() + [{"role": "user", "content": user_input}]
    else:
        new_input = [{"role": "user", "content": user_input}]
    print("\nUser Input:")
    print_markdown(user_input)

    
   # This is the core AI agent execution step. It runs your agent with the new_input.

    result = await Runner.run(starting_agent = agent, input = new_input)
    print("\nAssistant:")
    print_markdown(result.final_output)


User Input:


whats the color of the sky


Assistant:


The color of the sky typically appears blue during the day due to the scattering of sunlight by the Earth's atmosphere. This scattering is more efficient at shorter wavelengths, which corresponds to the blue part of the light spectrum. However, the sky can appear in various colors at different times, such as red, orange, or purple during sunrise and sunset.

In [17]:
# Let's view the list of tools that have been called
for i in result.to_input_list():
    for key in i.keys():
        if key == 'arguments':
            print("Tool: ", i['name'])
            print("Arguments: ", i['arguments'])
